# 00. 강사용 사전점검

수업 전에만 실행하는 노트북입니다. 학생에게 보여 줄 메인 시연은 01번부터 시작합니다.

저장된 실측 run, 펌웨어, 분석 환경을 먼저 확인합니다. 현재 보드가 없어도 저장 데이터 검사는 모두 실행됩니다. 내일 CW-Lite를 연결한 뒤 첫 셀을 다시 실행하면 USB 인식 상태도 확인할 수 있습니다.

## 저장 데이터와 USB 확인

`data/demo_runs.json`에는 수업에서 사용할 세 개의 실측 run이 고정되어 있습니다. 새 run을 자동으로 고르지 않으므로 수업 도중 결과가 바뀌지 않습니다.

In [1]:
import json
from pathlib import Path

import chipwhisperer as cw
import numpy as np

from cw_sca_tutorial.demo import demo_run, repository_root

ROOT = repository_root()
RUNS = {
    "password": demo_run("password_spa"),
    "tvla": demo_run("aes_tvla"),
    "cpa": demo_run("aes_cpa"),
}

devices = cw.list_devices()
print("Connected ChipWhisperer devices:", devices)
for name, run in RUNS.items():
    metadata = json.loads((run / "metadata.json").read_text())
    print(f"{name:8s}  {run.name}  state={metadata['state']}")

Connected ChipWhisperer devices: [{'name': 'ChipWhisperer-Lite', 'sn': '50203220594a48303130373132323034', 'hw_loc': (2, 6)}]
password  20261022T124300Z-password-spa  state=complete
tvla      20261022T125656Z-aes-tvla  state=complete
cpa       20261022T124611Z-aes-cpa  state=complete


## 신호와 데이터 모양 확인

비밀번호 preflight는 25 dB에서 50 ADC count 이상의 폭이 있어야 합니다. TVLA는 fixed/random 각 500개, CPA는 500개 파형을 사용합니다. 아래 값이 다르면 메인 노트북을 열기 전에 원인을 확인하십시오.

In [2]:
password_preflight = np.load(RUNS["password"] / "preflight_trace.npy")
password_span = float(np.ptp(password_preflight))
tvla = np.load(RUNS["tvla"] / "traces.npz")
cpa = np.load(RUNS["cpa"] / "traces.npz")

print(f"Password preflight peak-to-peak: {password_span:.1f} ADC counts")
print("TVLA shape:", tvla["traces"].shape, "groups:", np.bincount(tvla["groups"]))
print("CPA shape:", cpa["traces"].shape)

assert password_span >= 50
assert tvla["traces"].shape == (1000, 5000)
assert np.array_equal(np.bincount(tvla["groups"]), [500, 500])
assert cpa["traces"].shape == (500, 5000)
print("Pre-class checks passed.")

Password preflight peak-to-peak: 608.0 ADC counts
TVLA shape: (1000, 5000) groups: [500 500]
CPA shape: (500, 5000)
Pre-class checks passed.


## 내일 보드를 연결한 뒤

터미널에서 다음 진단을 한 번 실행합니다.

```bash
python scripts/run_hardware_capture.py diagnostic   --platform CWLITEARM   --target-identification STM32F303
```

25 dB 파형 폭이 한 자릿수라면 분석을 시작하지 말고 `MEASURE` SMA와 shunt 경로를 확인합니다. 이 장치에서는 정상 연결 시 약 618 count였습니다.